In [0]:
%sql

USE CATALOG bootcamp;

show schemas

In [0]:
%sql
create schema bronze

In [0]:
%sql

create volume landing.archivos
comment "Volume to storage bootcamp data";
show volumes in bootcamp.landing;

In [0]:
%sql

select * from read_files(
    '/Volumes/bootcamp/landing/archivos/properties_raw.csv',
    format=>'csv',
    header=>true
)

In [0]:
%sql
CREATE TABLE bronze.properties_bronze
    select * from read_files(
        '/Volumes/bootcamp/landing/archivos/properties_raw.csv',
        format=>'csv',
        header=>true
    )
    where url LIKE 'https%'

In [0]:
%sql

use bronze;
DESCRIBE properties_bronze

In [0]:
%sql

use samples.nyctaxi;

describe trips

In [0]:
%sql

with zonas as (
    select pickup_zip,count(*) as amount_trips, round(avg(fare_amount),2) as avg_fare, round(avg(trip_distance),2) as avg_distance
    from trips
     WHERE pickup_zip IS NOT NULL
    AND fare_amount > 0
    AND trip_distance > 0
    group by pickup_zip
)

select pickup_zip, amount_trips,avg_fare,  avg_distance from zonas
where amount_trips > 100


In [0]:
%sql

with avg_hora as (
    select hour(tpep_pickup_datetime) as hour,count(*) as Total_trips, round(avg(fare_amount),2) as hour_avg_fare
    from trips
    where fare_amount > 0
   
    group by hour(tpep_pickup_datetime)
),

general_avg as (

    select round(avg(fare_amount),2) as general_fare from trips
    where fare_amount > 0
   
)

select 
h.hour
,h.Total_trips
,h.hour_avg_fare
,g.general_fare
,round(((h.hour_avg_fare - g.general_fare)),2) as difference
,round(((h.hour_avg_fare - g.general_fare)/g.general_fare)*100,2) as difference_percentage 
from avg_hora h
cross join general_avg g
order by h.hour



In [0]:
%sql

with valid_trips as (
    select *
    from trips
    where fare_amount > 0
    and trip_distance > 0
    AND trip_distance IS NOT NULL
    AND fare_amount IS NOT NULL
)

select 
count(*) as total_trips
,min(fare_amount) as min_fare
,max(fare_amount) as max_fare
,round (avg(fare_amount),2) as avg_fare
,round(median(fare_amount),2) as median_fare
,min(trip_distance) as min_Distance
,max(trip_distance) as max_Distance
,round(avg(trip_distance),2) as avg_Distance
,median(trip_distance) as median_Distance

from valid_trips
    




In [0]:
%sql


select row_number()over(order by fare_amount desc) as ranking
,tpep_pickup_datetime,
round(trip_distance,2)  as trip_distance,
round(fare_amount,2) as fare_amount
from trips
where fare_amount > 0
order by ranking
limit 20

In [0]:
%sql

with zone_trips as (
select pickup_zip,count(*) as amount_trips
from trips
group by pickup_zip
)

select pickup_zip,amount_trips,
row_number() over(order by amount_trips desc) as row_number,
rank() over(order by amount_trips desc) as rank,
dense_rank() over(order by amount_trips desc) as dense_rank

from zone_trips  

In [0]:
%sql
SELECT 
  ROW_NUMBER() OVER (ORDER BY fare_amount DESC) AS ranking,
  tpep_pickup_datetime AS fecha,
  ROUND(trip_distance, 2) AS distancia,
  ROUND(fare_amount, 2) AS tarifa
FROM samples.nyctaxi.trips
WHERE fare_amount > 0
ORDER BY fare_amount DESC
LIMIT 20;

In [0]:
%sql

select 
tpep_pickup_datetime,
pickup_zip,
trip_distance,
fare_amount,
round(avg(fare_amount) over (),2) as avg_fare,
round((fare_amount-avg_fare),2) as difference,
round((fare_amount-avg_fare)/avg_fare*100,2) as difference_percentage
from trips
where fare_amount > 0
ORDER BY tpep_pickup_datetime DESC
LIMIT 20;
   



In [0]:
%sql

select 
tpep_pickup_datetime,
pickup_zip,
trip_distance,
fare_amount,
round(avg(fare_amount) over (partition by pickup_zip),2) as avg_fare_per_zone,
round((fare_amount-avg_fare_per_zone),2) as difference,
round((fare_amount-avg_fare_per_zone)/avg_fare_per_zone*100,2) as difference_percentage
from trips
WHERE fare_amount > 0
  AND pickup_zip IS NOT NULL
ORDER BY pickup_zip, fare_amount DESC
LIMIT 50;

In [0]:
%sql

select
tpep_pickup_datetime,
fare_amount,
lag(fare_amount,1) over (order by tpep_pickup_datetime) as previous_fare,
ROUND(fare_amount - LAG(fare_amount) OVER (ORDER BY tpep_pickup_datetime), 2) AS diferencia_con_anterior
from trips
where fare_amount > 0
order by tpep_pickup_datetime
limit 20


In [0]:
%sql

select
tpep_pickup_datetime,fare_amount,
ROUND(SUM(fare_amount) OVER (
    ORDER BY tpep_pickup_datetime 
  ), 2) AS total_acumulado
from trips
where fare_amount>0
order by tpep_pickup_datetime
limit 20

In [0]:
%sql

select
tpep_pickup_datetime,fare_amount,
sum(fare_amount) over(order by tpep_pickup_datetime ROWS BETWEEN UNBOUNDED PRECEDING AND
CURRENT ROW) as running_total
from trips
where fare_amount>0
order by tpep_pickup_datetime
limit 20

In [0]:
%sql

with valid_trips as (
    select *
    from trips
    where fare_amount > 0
    and trip_distance > 0
    AND trip_distance IS NOT NULL
    AND fare_amount IS NOT NULL
),

zone_values as(
select 
pickup_zip
,count(*) as total_trips
,min(fare_amount)  as min_fare_zone
,max(fare_amount)    as max_fare_zone
,round(avg(fare_amount),2)  as avg_fare_zone
from valid_trips
group by pickup_zip
having count(*) >= 50
)

select *,
row_number() over (order by avg_fare_zone desc) as ranking
from zone_values
order by ranking
limit 10
